In [5]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.datasets import CelebA
from torchvision.models import resnet18, ResNet18_Weights
from torchmetrics.classification import BinaryAccuracy, BinaryAUROC


# Set random seed to ensure reproducible results
torch.manual_seed(42)

# Select CPU as execution device
device = torch.device("cpu")
print(f"Running model on: {device}")

# Training Hyperparameters
BATCH_SIZE = 64        # Number of samples processed per batch for stochastic gradient descent
LEARNING_RATE = 1e-4    # Learning rate for Adam optimizer
NUM_EPOCHS = 5          # Total training passes over the dataset. Not too many necesarry since ResNet is pre-trained
DATA_DIR = "./data"     # Data location

# Target classification label
TARGET_ATTR = "Smiling"

# The 10 facial attribute concepts in the project definition
CONCEPT_ATTRS = [
    "Mouth_Slightly_Open",
    "High_Cheekbones",
    "Chubby",
    "Narrow_Eyes",
    "Bags_Under_Eyes",
    "Big_Lips",
    "Big_Nose",
    "Pointy_Nose",
    "Bushy_Eyebrows",
    "Arched_Eyebrows",
]

# Cropping out border pixels and focusing on 64x64 pixel space for the facial images
transform = transforms.Compose([
    transforms.CenterCrop(178),
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
])

# Load full CelebA dataset splits
full_train_dataset = CelebA(root=DATA_DIR, split="train", target_type="attr", download=False, transform=transform)
full_test_dataset  = CelebA(root=DATA_DIR, split="test",  target_type="attr", download=False, transform=transform)

# Subsample dataset indices to speed up execution time on CPU
# Taking 5,000 samples for training and 1,000 samples for testing
train_indices = list(range(5000))
test_indices  = list(range(1000))
train_dataset = Subset(full_train_dataset, train_indices)
test_dataset  = Subset(full_test_dataset, test_indices)

# Retrieve column index for 'Smiling' target
target_idx = full_train_dataset.attr_names.index(TARGET_ATTR)

# Custom batch aggregation function to extract 'Smiling' target vector
# This is used because the CelebA includes large multidimensional images
# so we define a custom collate_fun to avoid returning all 40 attributes per 
# image and just the specific Smiling target label.
# Returns a tuple of the images and their corresponding target attribute
def collate_fn(batch):
    # Separate image tensors and attribute targets
    images, target_attrs = zip(*batch)
    
    # Stack images into batch tensor: (batch_size, 3, 64, 64)
    images = torch.stack(images, dim=0)
    
    # Extract raw target attribute values and map from (-1, 1) to binary float (0.0, 1.0)
    raw_targets = torch.stack([attr[target_idx] for attr in target_attrs])
    targets = (raw_targets == 1).float().unsqueeze(1)
    
    return images, targets

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Subsampled train set size: {len(train_dataset)} images")
print(f"Subsampled test set size:  {len(test_dataset)} images")

class BaselineClassifier(nn.Module):
    """
    Standard image classifier mapping input images directly to output target label y.
    Uses ImageNet-pretrained ResNet-18 backbone.
    """
    def __init__(self, num_classes=1, pretrained=True):
        super(BaselineClassifier, self).__init__()
        
        # Load ResNet-18 model with pretrained ImageNet weights
        weights = ResNet18_Weights.DEFAULT if pretrained else None
        resnet = resnet18(weights=weights)
        
        # Extract features dimension before the original fc layer (512 features)
        in_features = resnet.fc.in_features
        
        # Remove the final fully connected classification layer
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])
        
        # Single linear classification head mapping features directly to 'Smiling' target
        self.classifier = nn.Linear(in_features, num_classes)

    def forward(self, x):
        # Pass images through ResNet-18 backbone feature extractor
        features = self.backbone(x)
        
        # Flatten spatial feature map output to 1D feature vector per image
        features = torch.flatten(features, 1)
        
        # Compute output logits
        logits = self.classifier(features)
        
        return logits

# Initialize baseline model, loss function, and optimizer
model = BaselineClassifier(num_classes=1, pretrained=True).to(device)
criterion = nn.BCEWithLogitsLoss() #Combines both the sigmoid function wtih the binary cross-entropy function
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE) #Instead of SGD, Adam computes individual learning rate per parameter


for epoch in range(NUM_EPOCHS):
    model.train()  # Set model to training mode
    running_loss = 0.0
    
    for images, targets in train_loader:
        # Move inputs and targets to CPU device
        images, targets = images.to(device), targets.to(device)
        
        # Clear previous parameter gradients
        optimizer.zero_grad()
        
        # Forward pass: compute predictions
        logits = model(images)
        
        # Calculate loss
        loss = criterion(logits, targets)
        
        # Backward pass: compute loss gradients
        loss.backward()
        
        # Update model weights
        optimizer.step()
        
        # Accumulate loss scaled by batch size
        # Done because 5000 subsample with 64 batch size results in a final batch of only 8
        # So this adjusts for that possibility
        running_loss += loss.item() * images.size(0) 
        
    # Print average epoch loss per image
    epoch_loss = running_loss / len(train_dataset)
    print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}] - Loss: {epoch_loss:.4f}")

# Initialize torchmetrics evaluators for binary classification
test_acc_metric = BinaryAccuracy().to(device)
test_auroc_metric = BinaryAUROC().to(device)

model.eval()  # Set model to evaluation mode
with torch.no_grad():  # Disable gradient computation
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)
        
        # Forward pass
        logits = model(images)
        
        # Apply sigmoid to map raw logits to probabilities [0.0, 1.0]
        probs = torch.sigmoid(logits)
        
        # Update evaluators with predicted probabilities and targets
        test_acc_metric.update(probs, targets.int())
        test_auroc_metric.update(probs, targets.int())

# Compute final metric scores
final_acc = test_acc_metric.compute().item()
final_auroc = test_auroc_metric.compute().item()

# Print metrics as required in project prompt
print("\n--- Baseline Classifier Evaluation Report ---")
print(f"Test Accuracy: {final_acc * 100:.2f}%")
print(f"Test AUROC:    {final_auroc:.4f}")

Running model on: cpu
Subsampled train set size: 5000 images
Subsampled test set size:  1000 images

Starting model training...
Epoch [1/5] - Loss: 0.5238
Epoch [2/5] - Loss: 0.1746
Epoch [3/5] - Loss: 0.0462
Epoch [4/5] - Loss: 0.0164
Epoch [5/5] - Loss: 0.0091

--- Baseline Classifier Evaluation Report ---
Test Accuracy: 87.00%
Test AUROC:    0.9433
